# Silver Layer - Cross Domain
## Files: StatusType.txt, TaxRate.txt, Industry.txt, TradeType.txt

###### Author: Prajwol Regmi

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import current_timestamp, col, row_number, lit
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime

# ─── CONFIG ──────────────────────────────────────────────────────────────
team_name = "team_lemma"
catalog_name = f"charles_schwab_retailbrokerage_dev_{team_name}"
bronze_schema = "bronze"
silver_schema = "silver"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {silver_schema}")

In [0]:
# ─── TABLE CONFIG: Cross-Domain Small Lookup Tables ──────────────────────
# B1 only, static reference data, no Gold tables needed
# Used as JOIN lookups downstream in Silver → Gold transformations

CROSS_LOOKUP_CONFIG = {
    "statustype": {
        "primary_key": "ST_ID",
        "source_table": f"{catalog_name}.{bronze_schema}.statustype",
        "target_table": f"{catalog_name}.{silver_schema}.statustype",
    },
    "taxrate": {
        "primary_key": "TX_ID",
        "source_table": f"{catalog_name}.{bronze_schema}.taxrate",
        "target_table": f"{catalog_name}.{silver_schema}.taxrate",
    },
    "industry": {
        "primary_key": "IN_ID",
        "source_table": f"{catalog_name}.{bronze_schema}.industry",
        "target_table": f"{catalog_name}.{silver_schema}.industry",
    },
    "tradetype": {
        "primary_key": "TT_ID",
        "source_table": f"{catalog_name}.{bronze_schema}.tradetype",
        "target_table": f"{catalog_name}.{silver_schema}.tradetype",
    },
}

In [0]:
# ─── REUSABLE SILVER PROCESSING FUNCTION ─────────────────────────────────
def process_bronze_to_silver_lookup(spark, table_name: str, config: dict) -> dict:
    """
    Processes a single Bronze lookup table to Silver:
    1. Reads from Bronze (ALL STRING columns)
    2. Deduplicates by primary key (latest _ingest_ts wins)
    3. Preserves all audit columns from Bronze
    4. Adds _load_ts for Silver layer audit
    5. Writes to Silver as Delta (overwrite)
    
    Returns a dict with recon metrics.
    """
    primary_key = config["primary_key"]
    source_table = config["source_table"]
    target_table = config["target_table"]
    
    # ── 1. Read from Bronze ──────────────────────────────────────────────
    bronze_df = spark.read.table(source_table)
    source_count = bronze_df.count()
    
    # ── 2. Deduplicate: ROW_NUMBER PARTITION BY PK ORDER BY _ingest_ts DESC ──
    # Latest record wins (most recent ingest timestamp)
    dedup_window = Window.partitionBy(primary_key).orderBy(col("_ingest_ts").desc())
    
    deduped_df = (
        bronze_df
        .withColumn("_row_num", row_number().over(dedup_window))
        .filter(col("_row_num") == 1)
        .drop("_row_num")
    )
    
    # ── 3. Add Silver audit column ───────────────────────────────────────
    # Carry forward _run_id from Bronze (do NOT generate new one)
    # Add _load_ts as Silver layer timestamp
    silver_df = deduped_df.withColumn("_load_ts", current_timestamp())
    
    # ── 4. Write to Silver ────────────────────────────────────────────────
    # Overwrite mode for idempotent full-refresh of static lookup tables
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)
    
    # ── 5. Validate target count ─────────────────────────────────────────
    target_count = spark.read.table(target_table).count()
    
    # Get the carried _run_id for lineage tracking
    carried_run_id = silver_df.select("_run_id").first()[0]
    
    print(f"  ✓ {table_name}: source={source_count} → deduped/silver={target_count} | run_id={carried_run_id}")
    
    return {
        "table_name": table_name,
        "source_count": source_count,
        "target_count": target_count,
        "carried_run_id": carried_run_id,
        "status": "SUCCESS"
    }

In [0]:
# ─── EXECUTE BRONZE → SILVER FOR ALL 4 LOOKUP TABLES ─────────────────────
print("="*60)
print("BRONZE → SILVER: Cross-Domain Small Lookup Tables")
print("="*60)

recon_results = []

for table_name, config in CROSS_LOOKUP_CONFIG.items():
    try:
        print(f"\nProcessing: {table_name}")
        result = process_bronze_to_silver_lookup(spark, table_name, config)
        recon_results.append(result)
    except Exception as e:
        print(f"  ✗ {table_name}: ERROR - {str(e)}")
        recon_results.append({
            "table_name": table_name,
            "source_count": None,
            "target_count": None,
            "carried_run_id": None,
            "status": f"ERROR: {str(e)[:200]}"
        })

print(f"\n{'='*60}")
print(f"Processing complete: {sum(1 for r in recon_results if r['status'] == 'SUCCESS')}/{len(recon_results)} tables succeeded")
print(f"{'='*60}")

In [0]:
# ─── OPERATIONS LOGGING: Reconciliation + Audit ──────────────────────────
# Follows the exact standard from landing_to_bronze pattern

for result in recon_results:
    if result["status"] == "SUCCESS":
        # 1. Log pipeline reconciliation (source vs target counts)
        log_pipeline_recon(
            spark=spark,
            run_id=result["carried_run_id"],
            batch_id="Batch1",
            domain="CROSS",
            table_name=result["table_name"],
            source_layer="bronze",
            target_layer="silver",
            source_count=result["source_count"],
            target_count=result["target_count"]
        )
        
        # 2. Log audit event (write operation)
        log_audit_event(
            spark=spark,
            run_id=result["carried_run_id"],
            batch="Batch1",
            layer="silver",
            table_name=result["table_name"],
            operation="OVERWRITE",
            rows_affected=result["target_count"]
        )

print("Operations logging complete.")

# ─── DISPLAY RECONCILIATION SUMMARY ──────────────────────────────────────
recon_df = spark.createDataFrame([
    Row(
        table=r["table_name"],
        source_bronze_count=r["source_count"],
        target_silver_count=r["target_count"],
        run_id=r["carried_run_id"],
        status=r["status"]
    ) for r in recon_results
])

display(recon_df)

In [0]:
# ─── VERIFY SILVER TABLES ────────────────────────────────────────────────
print("silver.statustype (6 rows expected)")
display(spark.read.table(f"{catalog_name}.{silver_schema}.statustype"))

print("\nsilver.taxrate (320 rows expected)")
display(spark.read.table(f"{catalog_name}.{silver_schema}.taxrate"))

print("\nsilver.industry (102 rows expected)")
display(spark.read.table(f"{catalog_name}.{silver_schema}.industry"))

print("\nsilver.tradetype (5 rows expected)")
display(spark.read.table(f"{catalog_name}.{silver_schema}.tradetype"))